In [66]:
import geopandas as gpd
import pandas as pd 
import sys
from skimage import measure
from osgeo import gdal

origin = '/workspace/'
sys.path.append('/media/')

from FieldWaterUseTools.FuncBox.DICT_LIST import EXCLUDE_LIST

states = ['Brandenburg', 'Niedersachsen', 'MV', 'NRW', 'Saarland']
state_folders = ['BRB', 'LSA', 'MV', 'NRW', 'SL']
random_seed = 42
thresh = 0.5

thuenen_path = f"{origin}fields/09_Thuenen_field_maps/"
thuenen_mod = 'CTM_GER_2023_seg_v201'
thuenen_file = f"{thuenen_path}{thuenen_mod}.gpkg"

state = 'Brandenburg'
model = 'FromScratch_dilate_T'
year = '2023'
mask = 'UnMasked'#'ThuenenMasked'
textbound = f"ext_03_bound_01"
polygonized_file = f"{origin}fields/07_Polygonized/{state}/{model}/{year}/{mask}/{mask}_{textbound}_10m.gpkg"
iacs_file = f"{origin}fields/01_IACS/1_Polygons/{state_folders[states.index(state)]}/GSA-DE_{state_folders[states.index(state)]}-{year}.geoparquet"

iacs_raster = f"{origin}fields/01_IACS/4_Crop_mask/{state_folders[states.index(state)]}/{year}/IACS_{state_folders[states.index(state)]}_{year}_cropMask_cropMask_lines_touch_false_crop_touch_false_linecrop.tif"
segm_raster = f"{origin}fields/06_Segmentation/{state}/{model}/{year}/{mask}/vrt/{mask}_{textbound}.vrt"

In [ ]:
# raster comparison
# label  iacs data
ds = gdal.Open(iacs_raster)
iacs_ras = ds.GetRasterBand(1).ReadAsArray()
iacs_labelled = measure.label(iacs_ras, background=0, connectivity=1)

ds = gdal.Open(segm_raster)
# here we need the relabelled version --> write away in 07_Polygonize

In [ ]:
print(iacs_labelled.shape)
print()

(27000, 30000)

In [ ]:
# vector comparison

In [58]:
our_polygons = gpd.read_file(polygonized_file)
# thuenen_polygons = gpd.read_file(thuenen_file)
# min_x, min_y, max_x, max_y = our_polygons.total_bounds
# offset = 3000 # in order to avoid cut off artefacts at potential boundary when comparing polygons
# thuenen_subset = thuenen_polygons.cx[(min_x - offset) : max_x, (min_y - offset) : (max_y + offset)].copy()
# thuenen_subset.to_file(f"{thuenen_path}{thuenen_mod}_subset_{state}.gpkg", driver="GPKG")
thuenen_subset = gpd.read_file(f"{thuenen_path}{thuenen_mod}_subset_{state}.gpkg")
iacs_polygons = gpd.read_parquet(iacs_file)

# rename IDs to prevent confusion when intersecting
our_polygons['OUR_FIELD_ID'] = our_polygons['FieldID']
our_polygons = our_polygons.drop(columns=['FieldID'])
our_polygons['OUR_AREA_ha'] = our_polygons.geometry.area / 10000

thuenen_subset['THUENEN_FIELD_ID'] = thuenen_subset['id']
thuenen_subset['THUENEN_AREA_ha'] = thuenen_subset['area_ha']
thuenen_subset = thuenen_subset.drop(columns=['id', 'area_ha'])

iacs_polygons['IACS_FIELD_ID'] = iacs_polygons['field_id']
iacs_polygons['IACS_AREA_ha'] = iacs_polygons['field_size']
iacs_polygons = iacs_polygons.drop(columns=['field_size', 'field_id'])

################################### 
### There is an issue with our polygons. In the polygonization process, some fields at the border of states are cut into multiple smaller fields with the same id.
### in the scope of validation they can be ignored, but the behaviour should be fixed  !!!!
duplicates = our_polygons[our_polygons["OUR_FIELD_ID"].isin(our_polygons["OUR_FIELD_ID"][our_polygons["OUR_FIELD_ID"].duplicated()])].sort_values("OUR_FIELD_ID")
kill_IDs = duplicates['OUR_FIELD_ID'].unique().tolist()
our_polygons_filtered = our_polygons[~our_polygons['OUR_FIELD_ID'].isin(kill_IDs)].copy()

In [59]:
# check projections
if thuenen_subset.crs != iacs_polygons.crs:
    print('reprojecting thuenen')
    thuenen_subset = thuenen_subset.to_crs(iacs_polygons.crs)
else:
    print('thuenen and iacs polygons have same CRS')

if our_polygons_filtered.crs != iacs_polygons.crs:
    print('reprojecting our polygons')
    our_polygons_filtered = our_polygons_filtered.to_crs(iacs_polygons.crs)
else:
    print('ours and iacs polygons have same CRS')

# draw a sample from IACS after subsetting to 5 pixel threshold and excluding polygons with attributes listed on EXCLUDE_LIST
iacs_polygons_filtered = iacs_polygons[~iacs_polygons['EC_hcat_n'].isin(EXCLUDE_LIST)].copy()
iacs_polygons_filtered = iacs_polygons_filtered[iacs_polygons_filtered['IACS_AREA_ha'] > 0.05] # fields size is in ha -> should be larger than 5 pixel
iacs_polygons_filtered['decile'] = pd.qcut(iacs_polygons_filtered['IACS_AREA_ha'], q=10, labels=False)

n_per_decile = 1500

iacs_sample = (
    iacs_polygons_filtered
    .groupby('decile', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), n_per_decile), random_state=random_seed), include_groups=False)
    .reset_index(drop=True)
)

thuenen and iacs polygons have same CRS
ours and iacs polygons have same CRS


In [60]:
# intersect our polygons with sample with IACS
intersections_IACS_OURS = gpd.overlay(our_polygons_filtered, iacs_sample, how='intersection')
intersections_IACS_OURS['OVERLAP_AREA_ha'] = intersections_IACS_OURS.geometry.area / 10000
intersections_IACS_OURS = intersections_IACS_OURS.drop(columns=['crop_code', 'crop_name', 'EC_trans_n']) # make it in ipynb more readable

# calculate ratio of overlap (in regards to IACS and to our polygons) and keep the intersecting polygons with highest overlap
intersections_IACS_OURS['OVERLAP_AREA_with_IACS_relative'] = intersections_IACS_OURS['OVERLAP_AREA_ha'] / intersections_IACS_OURS['IACS_AREA_ha']
intersections_IACS_OURS.loc[intersections_IACS_OURS['OVERLAP_AREA_with_IACS_relative'] > 1,'OVERLAP_AREA_with_IACS_relative'] = 1

intersections_IACS_OURS['OVERLAP_AREA_with_OURS_relative'] = intersections_IACS_OURS['OVERLAP_AREA_ha'] / intersections_IACS_OURS['OUR_AREA_ha']
intersections_IACS_OURS.loc[intersections_IACS_OURS['OVERLAP_AREA_with_OURS_relative'] > 1,'OVERLAP_AREA_with_OURS_relative'] = 1

intersections_IACS_OURS.to_file(f"{thuenen_path}TEMP/intersects_ours.gpkg", driver="GPKG")

In [62]:
best_match_ours = intersections_IACS_OURS[(intersections_IACS_OURS['OVERLAP_AREA_with_IACS_relative'] > thresh) & 
                                          (intersections_IACS_OURS['OVERLAP_AREA_with_OURS_relative'] > thresh)]

In [30]:
best_match_ours['OUR_FIELD_ID'].duplicated().sum()

406

In [ ]:
# idx = intersections_IACS_OURS.groupby("IACS_FIELD_ID")["OVERLAP_AREA_with_IACS_relative"].idxmax()
# best_match_ours = intersections_IACS_OURS.loc[idx].copy()
# best_match_ours = best_match_ours[best_match_ours["OVERLAP_AREA_with_IACS_relative"] > thresh]

# best_match_ours.to_file(f"{thuenen_path}TEMP/best_match_ours.gpkg", driver="GPKG")

In [63]:
# compute union for best matches
best_match_ours_df = best_match_ours.drop(columns=['geometry']).copy()
merged = best_match_ours_df.merge(our_polygons[['OUR_FIELD_ID', 'geometry']], on='OUR_FIELD_ID', how='left')
merged = merged.rename(columns={"geometry": "geometry_OURS"})

merged = merged.merge(
    iacs_sample[[ "IACS_FIELD_ID", "geometry" ]].rename(columns={"geometry": "geometry_IACS"}),
    on="IACS_FIELD_ID",
    how="left"
)

merged["geometry"] = merged.apply(
    lambda row: row["geometry_OURS"].union(row["geometry_IACS"]),
    axis=1
)

union_IACS_OURS = gpd.GeoDataFrame(merged, geometry="geometry", crs=our_polygons_filtered.crs)
union_IACS_OURS = union_IACS_OURS.drop(columns=['geometry_OURS', 'geometry_IACS'])
union_IACS_OURS['UNION_AREA_ha'] = union_IACS_OURS.geometry.area / 10000
union_IACS_OURS['IoU'] = union_IACS_OURS['OVERLAP_AREA_ha'] / union_IACS_OURS['UNION_AREA_ha']
union_IACS_OURS.to_file(f"{thuenen_path}TEMP/union_IACS_OURS.gpkg", driver="GPKG")

In [51]:
# intersect thuenen polygons with sample with IACS
intersections_IACS_TH = gpd.overlay(thuenen_subset, iacs_sample, how='intersection')
intersections_IACS_TH['OVERLAP_AREA_ha'] = intersections_IACS_TH.geometry.area / 10000
intersections_IACS_TH = intersections_IACS_TH.drop(columns=['crop_code', 'crop_name', 'EC_trans_n']) # make it in ipynb more readable

# calculate ratio of overlap (in regards to IACS and to our polygons) and keep the intersecting polygons with highest overlap
intersections_IACS_TH['OVERLAP_AREA_with_IACS_relative'] = intersections_IACS_TH['OVERLAP_AREA_ha'] / intersections_IACS_TH['IACS_AREA_ha']
intersections_IACS_TH.loc[intersections_IACS_TH['OVERLAP_AREA_with_IACS_relative'] > 1,'OVERLAP_AREA_with_IACS_relative'] = 1

intersections_IACS_TH['OVERLAP_AREA_with_TH_relative'] = intersections_IACS_TH['OVERLAP_AREA_ha'] / intersections_IACS_TH['THUENEN_AREA_ha']
intersections_IACS_TH.loc[intersections_IACS_TH['OVERLAP_AREA_with_TH_relative'] > 1,'OVERLAP_AREA_with_TH_relative'] = 1

intersections_IACS_TH.to_file(f"{thuenen_path}TEMP/intersects_th.gpkg", driver="GPKG")

/tmp/ipykernel_839842/3603400930.py:2: UserWarning: `keep_geom_type=True` in overlay resulted in 3 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  intersections_IACS_TH = gpd.overlay(thuenen_subset, iacs_sample, how='intersection')


In [52]:
best_match_TH = intersections_IACS_TH[(intersections_IACS_TH['OVERLAP_AREA_with_IACS_relative'] > thresh) & 
                        (intersections_IACS_TH['OVERLAP_AREA_with_TH_relative'] > thresh)]

In [ ]:
# idx = intersections_IACS_TH.groupby("IACS_FIELD_ID")["OVERLAP_AREA_with_IACS_relative"].idxmax()
# best_match_TH = intersections_IACS_TH.loc[idx].copy()
# best_match_TH = best_match_TH[best_match_TH["OVERLAP_AREA_with_IACS_relative"] > thresh]

# best_match_TH.to_file(f"{thuenen_path}TEMP/best_match_ours.gpkg", driver="GPKG")

In [53]:
# compute union for best matches
best_match_TH_df = best_match_TH.drop(columns=['geometry']).copy()
merged = best_match_TH_df.merge(thuenen_subset[['THUENEN_FIELD_ID', 'geometry']], on='THUENEN_FIELD_ID', how='left')
merged = merged.rename(columns={"geometry": "geometry_TH"})

merged = merged.merge(
    iacs_sample[[ "IACS_FIELD_ID", "geometry" ]].rename(columns={"geometry": "geometry_IACS"}),
    on="IACS_FIELD_ID",
    how="left"
)

merged["geometry"] = merged.apply(
    lambda row: row["geometry_TH"].union(row["geometry_IACS"]),
    axis=1
)

union_IACS_TH = gpd.GeoDataFrame(merged, geometry="geometry", crs=thuenen_subset.crs)
union_IACS_TH = union_IACS_TH.drop(columns=['geometry_TH', 'geometry_IACS'])
union_IACS_TH['UNION_AREA_ha'] = union_IACS_TH.geometry.area / 10000
union_IACS_TH['IoU'] = union_IACS_TH['OVERLAP_AREA_ha'] / union_IACS_TH['UNION_AREA_ha']
union_IACS_TH.to_file(f"{thuenen_path}TEMP/union_IACS_THUENEN.gpkg", driver="GPKG")

In [64]:
print(f"mean of our polygons: {union_IACS_OURS['IoU'].mean()}")
print(f"mean of gideons polygons: {union_IACS_TH['IoU'].mean()}")

mean of our polygons: 0.8044645973282094
mean of gideons polygons: 0.7345826693057487
